# Patent Emerging Radar — Training Only

Starts from pre-computed cluster files (skips re-clustering).

**Input dataset:** upload `cluster_panel.csv`, `cluster_labels.csv`, `cluster_centroids.pkl`, `cluster_titles.pkl` as a Kaggle dataset and attach it.

**Output:** `train_output.zip` — models + scalers + leaderboards.

In [ ]:
import tensorflow as tf
print(f'TensorFlow {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')


In [ ]:
import json, pathlib, pickle, warnings, zipfile, datetime
from collections import defaultdict

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import spearmanr
from tqdm.auto import tqdm
import joblib

warnings.filterwarnings('ignore')

KAGGLE_INPUT = pathlib.Path('/kaggle/input')
OUT_DIR      = pathlib.Path('/kaggle/working')
(OUT_DIR / 'models').mkdir(exist_ok=True)

# Find cluster files (search recursively under /kaggle/input)
CLUSTER_DIR = None
for p in KAGGLE_INPUT.rglob('cluster_panel.csv'):
    CLUSTER_DIR = p.parent
    print(f'Found cluster files at: {CLUSTER_DIR}')
    break

if CLUSTER_DIR is None:
    all_files = [str(p) for p in KAGGLE_INPUT.rglob('*') if p.is_file()][:30]
    raise FileNotFoundError(
        f'cluster_panel.csv not found under /kaggle/input/\n'
        f'Files found: {all_files}\n'
        'Attach the dataset containing your 4 cluster files.'
    )


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
WINDOW   = 8
HORIZONS = [8, 12]
BATCH    = 64
EPOCHS   = 80
LR       = 3e-4

TRAIN_END = (2017, 4)
VAL_END   = (2020, 4)
TEST_END  = {8: (2023, 4), 12: (2022, 4)}

PRIMARY_H = HORIZONS[0]

print('Config ready.')
print(f'Horizons: {[f"{h//4}yr" for h in HORIZONS]}')


In [ ]:
# ── Load cluster files ─────────────────────────────────────────────────────────
panel_full = pd.read_csv(CLUSTER_DIR / 'cluster_panel.csv')

labels_df     = pd.read_csv(CLUSTER_DIR / 'cluster_labels.csv')
cluster_labels = dict(zip(labels_df['cluster_id'], labels_df['auto_label']))

with open(CLUSTER_DIR / 'cluster_centroids.pkl', 'rb') as f:
    cluster_centroids = pickle.load(f)
with open(CLUSTER_DIR / 'cluster_titles.pkl', 'rb') as f:
    cluster_titles = pickle.load(f)

print(f'Panel  : {panel_full.cluster_id.nunique()} clusters x {panel_full.groupby(["year","quarter"]).ngroups} quarters')
print(f'Labels : {len(cluster_labels)} entries')
print(f'Sample labels:')
for cid in list(cluster_labels)[:5]:
    print(f'  C{cid:03d}  {cluster_labels[cid]}')


In [ ]:
# ── Feature engineering ────────────────────────────────────────────────────────
ALL_FEATURES = [
    'log_count', 'count_share',
    'velocity', 'acceleration', 'jerk',
    'mom_4q', 'mom_8q',
    'roll_mean_4q', 'roll_std_4q', 'roll_mean_8q', 'roll_std_8q',
    'above_trend', 'macd', 'z_score',
    'global_rank_pctl', 'log_cumsum', 'sin_q', 'cos_q',
]
TARGET = 'log_count'

def make_features(panel: pd.DataFrame) -> pd.DataFrame:
    p = panel.copy().sort_values(['cluster_id', 'date'])
    p['log_count']   = np.log1p(p['count'])
    global_q         = p.groupby('date')['count'].transform('sum').replace(0, np.nan)
    p['count_share'] = p['count'] / global_q
    grp = p.groupby('cluster_id')['log_count']
    p['velocity']     = grp.diff().fillna(0)
    p['acceleration'] = p.groupby('cluster_id')['velocity'].diff().fillna(0)
    p['jerk']         = p.groupby('cluster_id')['acceleration'].diff().fillna(0)
    p['mom_4q'] = p.groupby('cluster_id')['log_count'].diff(4).fillna(0)
    p['mom_8q'] = p.groupby('cluster_id')['log_count'].diff(8).fillna(0)
    def roll(series, w, fn):
        return series.groupby(p['cluster_id']).transform(
            lambda x: getattr(x.rolling(w, min_periods=1), fn)()
        )
    p['roll_mean_4q'] = roll(p['log_count'], 4, 'mean')
    p['roll_std_4q']  = roll(p['log_count'], 4, 'std').fillna(0)
    p['roll_mean_8q'] = roll(p['log_count'], 8, 'mean')
    p['roll_std_8q']  = roll(p['log_count'], 8, 'std').fillna(0)
    p['above_trend']  = p['log_count'] - p['roll_mean_8q']
    p['macd']         = p['roll_mean_4q'] - p['roll_mean_8q']
    p['z_score']      = p['above_trend'] / (p['roll_std_8q'] + 1e-6)
    p['global_rank_pctl'] = p.groupby('date')['log_count'].rank(pct=True)
    p['log_cumsum']   = np.log1p(p.groupby('cluster_id')['count'].cumsum())
    p['sin_q']        = np.sin(2 * np.pi * p['quarter'] / 4)
    p['cos_q']        = np.cos(2 * np.pi * p['quarter'] / 4)
    return p.fillna(0)

df = make_features(panel_full)
print(f'Features: {len(ALL_FEATURES)}  |  Clusters: {df.cluster_id.nunique()}  |  Rows: {len(df):,}')


In [ ]:
# ── build_samples ──────────────────────────────────────────────────────────────
def build_samples(df, split, horizon):
    Xs, ys, metas = [], [], []
    for cid, grp in df.groupby('cluster_id'):
        grp  = grp.sort_values('date').reset_index(drop=True)
        vals = grp[ALL_FEATURES].values
        n    = len(vals)
        for i in range(WINDOW, n - horizon + 1):
            anchor_row = grp.iloc[i - 1]
            ay, aq     = int(anchor_row['year']), int(anchor_row['quarter'])
            in_train   = (ay, aq) <= TRAIN_END
            in_val     = (not in_train) and (ay, aq) <= VAL_END
            in_test    = not in_train and not in_val and (ay, aq) <= TEST_END[horizon]
            if split == 'train' and not in_train: continue
            if split == 'val'   and not in_val:   continue
            if split == 'test'  and not in_test:  continue
            ys.append(float(grp.iloc[i + horizon - 1][TARGET]) - float(grp.iloc[i - 1][TARGET]))
            Xs.append(vals[i - WINDOW:i])
            metas.append({'cluster_id': cid, 'anchor_year': ay, 'anchor_q': aq})
    return (np.array(Xs, dtype=np.float32),
            np.array(ys, dtype=np.float32),
            metas)

X_tr0, y_tr0, _ = build_samples(df, 'train', PRIMARY_H)
X_v0,  y_v0,  _ = build_samples(df, 'val',   PRIMARY_H)
print(f'Primary ({PRIMARY_H//4}yr) — train: {X_tr0.shape}  val: {X_v0.shape}')
print(f'y_train  mean={y_tr0.mean():.3f}  std={y_tr0.std():.3f}')


In [ ]:
# ── Model + training utils ─────────────────────────────────────────────────────
from tensorflow import keras

T, F = WINDOW, len(ALL_FEATURES)

class TQDMEpochBar(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self._pbar = tqdm(total=self.params['epochs'], desc='  epochs', unit='ep', leave=True)
    def on_epoch_end(self, epoch, logs=None):
        self._pbar.update(1)
        self._pbar.set_postfix({
            'val_loss': f"{logs.get('val_loss', 0):.4f}",
            'val_mae':  f"{logs.get('val_mae',  0):.4f}",
        })
    def on_train_end(self, logs=None):
        self._pbar.close()

def build_model(arch: str):
    inp = keras.Input(shape=(T, F), name='input')
    RNN = keras.layers.GRU if arch == 'gru' else keras.layers.LSTM
    x   = RNN(64, return_sequences=True, name=f'{arch}_1')(inp)
    x   = keras.layers.Dropout(0.2)(x)
    x   = RNN(32, name=f'{arch}_2')(x)
    x   = keras.layers.Dropout(0.2)(x)
    out = keras.layers.Dense(1, name='output')(x)
    m   = keras.Model(inp, out, name=arch)
    m.compile(optimizer=keras.optimizers.Adam(LR),
              loss=keras.losses.Huber(delta=1.0),
              metrics=['mae'])
    return m

def train_model(model, name, X_tr, y_tr, X_v, y_v):
    stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
    ckpt  = str(OUT_DIR / 'models' / f'{name}_{stamp}.keras')
    cbs = [
        TQDMEpochBar(),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=10,
                                      restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                          patience=5, verbose=0),
        keras.callbacks.ModelCheckpoint(ckpt, save_best_only=True, verbose=0),
    ]
    hist = model.fit(X_tr, y_tr, validation_data=(X_v, y_v),
                     epochs=EPOCHS, batch_size=BATCH,
                     callbacks=cbs, verbose=0)
    return hist, ckpt

build_model('gru').summary()


In [ ]:
# ── Train GRU + LSTM for each horizon ─────────────────────────────────────────
horizon_models = {}

for H in tqdm(HORIZONS, desc='Horizons', unit='horizon'):
    hlabel = f'{H//4}yr'
    print(f'\n{"="*55}\nHorizon {hlabel} ({H}Q)\n{"="*55}')

    X_tr, y_tr, _ = build_samples(df, 'train', H)
    X_v,  y_v,  _ = build_samples(df, 'val',   H)
    X_te, y_te, _ = build_samples(df, 'test',  H)

    sc = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr.reshape(-1, F)).reshape(-1, T, F)
    X_v_sc  = sc.transform(X_v.reshape(-1, F)).reshape(-1, T, F)
    X_te_sc = sc.transform(X_te.reshape(-1, F)).reshape(-1, T, F)
    (OUT_DIR / 'models').mkdir(exist_ok=True)
    joblib.dump(sc, OUT_DIR / 'models' / f'scaler_{hlabel}.pkl')

    print(f'  Samples — train: {len(y_tr):,}  val: {len(y_v):,}  test: {len(y_te):,}')

    gru  = build_model('gru')
    print('  Training GRU ...')
    hist_gru,  ckpt_gru  = train_model(gru,  f'gru_{hlabel}',  X_tr_sc, y_tr, X_v_sc, y_v)

    lstm = build_model('lstm')
    print('  Training LSTM ...')
    hist_lstm, ckpt_lstm = train_model(lstm, f'lstm_{hlabel}', X_tr_sc, y_tr, X_v_sc, y_v)

    gru_sp  = spearmanr(y_v, gru.predict(X_v_sc,  verbose=0).flatten())[0]
    lstm_sp = spearmanr(y_v, lstm.predict(X_v_sc, verbose=0).flatten())[0]
    best      = gru  if gru_sp >= lstm_sp else lstm
    best_arch = 'GRU' if gru_sp >= lstm_sp else 'LSTM'

    horizon_models[H] = {
        'gru': gru, 'lstm': lstm, 'best': best, 'best_arch': best_arch,
        'scaler': sc,
        'X_val': X_v,   'y_val': y_v,   'X_val_sc': X_v_sc,
        'X_test': X_te, 'y_test': y_te, 'X_test_sc': X_te_sc,
        'val_sp_gru': gru_sp, 'val_sp_lstm': lstm_sp,
        'ckpt_gru': ckpt_gru, 'ckpt_lstm': ckpt_lstm,
    }
    print(f'  Best: {best_arch}  val Spearman={max(gru_sp, lstm_sp):.4f}')


In [ ]:
# ── Val metrics + baselines ────────────────────────────────────────────────────
log_count_idx = ALL_FEATURES.index('log_count')
steps_arr     = np.arange(WINDOW, dtype=np.float32)
A_fit         = np.vstack([steps_arr, np.ones(WINDOW)]).T

for H, hd in horizon_models.items():
    hlabel = f'{H//4}yr'
    y_v    = hd['y_val']
    X_v    = hd['X_val']

    y_persist = np.zeros_like(y_v)
    log_hist  = X_v[:, :, log_count_idx]
    slopes    = np.linalg.lstsq(A_fit, log_hist.T, rcond=None)[0][0]
    y_trend   = slopes * H

    gru_pred  = hd['gru'].predict(hd['X_val_sc'],  verbose=0).flatten()
    lstm_pred = hd['lstm'].predict(hd['X_val_sc'], verbose=0).flatten()

    rows = []
    for pred, name in [(gru_pred, 'GRU'), (lstm_pred, 'LSTM'),
                       (y_trend, 'Linear trend'), (y_persist, 'Persistence (0)')]:
        sp, _ = spearmanr(y_v, pred)
        rmse  = float(np.sqrt(mean_squared_error(y_v, pred)))
        mae   = float(mean_absolute_error(y_v, pred))
        rows.append({'Model': name, 'Spearman': round(sp, 4),
                     'RMSE': round(rmse, 4), 'MAE': round(mae, 4)})

    print(f'\n── {hlabel} val metrics ──')
    print(pd.DataFrame(rows).to_string(index=False))


In [ ]:
# ── Leaderboards ──────────────────────────────────────────────────────────────
leaderboards = {}

for H, hd in horizon_models.items():
    hlabel      = f'{H//4}yr'
    rows        = []
    cluster_ids = df.cluster_id.unique()
    for cid in tqdm(cluster_ids, desc=f'Leaderboard {hlabel}', unit='cluster'):
        g = df[df.cluster_id == cid].sort_values('date')
        if len(g) < WINDOW: continue
        recent    = g.tail(WINDOW)[ALL_FEATURES].values.astype(np.float32)
        recent_sc = hd['scaler'].transform(recent).reshape(1, T, F)
        pred_delta = float(hd['best'].predict(recent_sc, verbose=0)[0][0])
        last_log   = float(g.iloc[-1]['log_count'])
        rows.append({
            'cluster_id':         cid,
            'auto_label':         cluster_labels.get(cid, f'cluster_{cid}'),
            'last_log_count':     round(last_log, 3),
            'forecast_log_count': round(last_log + pred_delta, 3),
            'predicted_growth':   round(pred_delta, 3),
            'last_date':          str(g.iloc[-1]['date']),
        })
    lb = (pd.DataFrame(rows)
          .sort_values('predicted_growth', ascending=False)
          .reset_index(drop=True))
    lb.to_csv(OUT_DIR / f'leaderboard_{hlabel}.csv', index=False)
    leaderboards[H] = lb
    print(f'{hlabel} — top 5:')
    print(lb[['auto_label', 'predicted_growth']].head(5).to_string(index=False))
    print()


In [ ]:
# ── 5yr linear extrapolation ──────────────────────────────────────────────────
extrap_rows = []
A_fit5      = np.vstack([np.arange(WINDOW, dtype=np.float32), np.ones(WINDOW)]).T

for cid in tqdm(df.cluster_id.unique(), desc='5yr extrapolation', unit='cluster'):
    g = df[df.cluster_id == cid].sort_values('date')
    if len(g) < WINDOW: continue
    log_hist = g.tail(WINDOW)['log_count'].values.astype(np.float32)
    slope    = np.linalg.lstsq(A_fit5, log_hist, rcond=None)[0][0]
    extrap   = float(slope * 20)
    last_log = float(g.iloc[-1]['log_count'])
    extrap_rows.append({
        'cluster_id':              cid,
        'auto_label':              cluster_labels.get(cid, f'cluster_{cid}'),
        'last_log_count':          round(last_log, 3),
        'extrapolated_growth_5yr': round(extrap, 3),
        'last_date':               str(g.iloc[-1]['date']),
    })

lb_5yr = (pd.DataFrame(extrap_rows)
          .sort_values('extrapolated_growth_5yr', ascending=False)
          .reset_index(drop=True))
lb_5yr.to_csv(OUT_DIR / 'leaderboard_5yr_extrap.csv', index=False)
print('5yr — top 5:')
print(lb_5yr[['auto_label', 'extrapolated_growth_5yr']].head(5).to_string(index=False))


In [ ]:
# ── Package output ────────────────────────────────────────────────────────────
out_zip = OUT_DIR / 'train_output.zip'

# Copy cluster files into output so everything is in one zip
import shutil
for fname in ['cluster_panel.csv', 'cluster_labels.csv',
              'cluster_centroids.pkl', 'cluster_titles.pkl']:
    src = CLUSTER_DIR / fname
    if src.exists():
        shutil.copy(src, OUT_DIR / fname)

files_to_zip = [
    (OUT_DIR / 'cluster_panel.csv',          'clusters/cluster_panel.csv'),
    (OUT_DIR / 'cluster_labels.csv',         'clusters/cluster_labels.csv'),
    (OUT_DIR / 'cluster_centroids.pkl',      'clusters/cluster_centroids.pkl'),
    (OUT_DIR / 'cluster_titles.pkl',         'clusters/cluster_titles.pkl'),
    (OUT_DIR / 'leaderboard_2yr.csv',        'clusters/leaderboard_2yr.csv'),
    (OUT_DIR / 'leaderboard_3yr.csv',        'clusters/leaderboard_3yr.csv'),
    (OUT_DIR / 'leaderboard_5yr_extrap.csv', 'clusters/leaderboard_5yr_extrap.csv'),
]
for p in sorted((OUT_DIR / 'models').iterdir()):
    files_to_zip.append((p, f'models/{p.name}'))

with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for src, arcname in tqdm(files_to_zip, desc='Packaging', unit='file'):
        if src.exists():
            zf.write(src, arcname)
            tqdm.write(f'  + {arcname}')
        else:
            tqdm.write(f'  ! missing: {src}')

print(f'\ntrain_output.zip  {out_zip.stat().st_size/1e6:.1f} MB')
print('Download from the Output tab, then:')
print('  unzip train_output.zip -d /home/jin/Documents/GITHUB/patent-emerging-radar/data/processed/')
